# Project 18: Rework Note Analysis, analisis Q1 sampai Q14
Notebook ini membaca CSV di `results/2-analysis` dan `results/1-data-preparation`. Data adalah simulasi yang dianonimkan. Klasifikasi memakai pengganti lokal (TF-IDF + Logistic Regression), bukan Claude API.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
R2 = Path('../2-analysis'); R1 = Path('../1-data-preparation')
pd.set_option('display.width', 160)

## Q1 dan H1: Pareto kategori

In [2]:
p = pd.read_csv(R2/'pareto.csv'); display(p); display(pd.read_csv(R2/'stats_pareto.csv'))
fig, ax = plt.subplots(figsize=(8,3)); ax.bar(p.category_code, p.n); ax.set_ylabel('catatan'); plt.show()

,rank,category_code,n,share,cum_share
0,1,C4,745,0.273495,0.273495
1,2,C5,594,0.218062,0.491557
2,3,C1,404,0.148311,0.639868
3,4,C7,278,0.102056,0.741924
4,5,C2,265,0.097283,0.839207
5,6,C6,193,0.070852,0.910059
6,7,C3,122,0.044787,0.954846
7,8,C8,93,0.034141,0.988987
8,9,C9,30,0.011013,1.000000


,k_kategori,kategori,n,cum_share,ci_low,ci_high,dalam_band,k_untuk_80
0,2,C4 C5,1339,0.491557,0.472808,0.510329,0,5
1,3,C4 C5 C1,1743,0.639868,0.621656,0.657686,0,5
2,4,C4 C5 C1 C7,2021,0.741924,0.725159,0.758007,0,5
3,5,C4 C5 C1 C7 C2,2286,0.839207,0.824936,0.852523,1,5


## Q2 dan H4: kategori per stasiun

In [3]:
display(pd.read_csv(R2/'pareto_station.csv').query('rank_in_station<=2')); display(pd.read_csv(R2/'stats_station_cell.csv').query('memenuhi_h4==1'))

,station_code,category_code,n,station_total,share,rank_in_station
0,ST-A,C4,108,477.0,0.226415,1
1,ST-A,C2,89,477.0,0.186583,2
9,ST-B,C4,118,458.0,0.257642,1
10,ST-B,C5,100,458.0,0.218341,2
18,ST-C,C4,252,492.0,0.512195,1
19,ST-C,C5,58,492.0,0.117886,2
27,ST-D,C7,100,406.0,0.246305,1
28,ST-D,C4,87,406.0,0.214286,2
36,ST-E,C5,198,441.0,0.448980,1
37,ST-E,C4,70,441.0,0.158730,2


,station_code,category_code,n,n_kategori,porsi_stasiun,porsi_seharusnya,rasio,p_mentah,p_holm,memenuhi_h4
0,ST-D,C7,100,278,0.359712,0.149046,2.413439,3.812669e-18,1.982588e-16,1
1,ST-E,C5,198,594,0.333333,0.161894,2.058957,1.014311e-24,5.375848e-23,1
2,ST-A,C2,89,265,0.335849,0.175110,1.917930,2.181326e-10,1.112476e-08,1
3,ST-C,C4,252,745,0.338255,0.180617,1.872778,6.518348e-25,3.519908e-23,1


## Q3 dan Q4: tren bulanan, shift, dan model

In [4]:
t = pd.read_csv(R2/'trend_month.csv'); display(t.pivot(index='month', columns='category_code', values='n')); display(pd.read_csv(R2/'stats_chi.csv')); display(pd.read_csv(R2/'stats_trend_flags.csv').query('lonjakan==1 or penurunan==1'))

category_code,C1,C2,C3,C4,C5,C6,C7,C8,C9
month,,,,,,,,,
2026-03,62,51,19,137,108,33,29,13,1
2026-04,74,48,25,108,82,36,48,12,4
2026-05,69,42,18,120,80,35,48,15,10
2026-06,64,41,20,133,109,35,48,19,4
2026-07,83,45,12,138,117,35,54,18,3
2026-08,52,38,28,109,98,19,51,16,8


,faktor,chi2,dof,p,cramers_v,n,sel_harapan_min,p_holm,signifikan
0,stasiun,466.317471,40,4.758533e-74,0.185034,2724,4.471366,1.903413e-73,1
1,shift,5.230077,8,7.327259e-01,0.043818,2724,14.581498,1.000000e+00,0
2,model,6.921123,16,9.747516e-01,0.035643,2724,9.504405,1.000000e+00,0
3,bulan,57.037813,40,3.930379e-02,0.064713,2724,4.614537,1.179114e-01,0


,month,category_code,n,share,share_rata,z,lonjakan,penurunan


## Q5 sampai Q7 dan H2, H3: ketepatan klasifikasi

In [5]:
display(pd.read_csv(R2/'eval_metrics.csv')); display(pd.read_csv(R2/'eval_per_class.csv')); display(pd.read_csv(R2/'eval_confusion_pairs.csv')); display(pd.read_csv(R2/'eval_compare.csv')); display(pd.read_csv(R2/'eval_abbrev.csv'))

,klasifikator,n,accuracy,acc_ci_low,acc_ci_high,macro_f1,f1_ci_low,f1_ci_high,salah
0,pengganti lokal (out-of-fold),269,0.992565,0.973301,0.997959,0.987701,0.956139,1,2
1,baseline kata kunci,269,1.000000,0.985921,1.000000,1.000000,0.888889,1,0


,category_code,support,precision,recall,f1,f1_baseline,recall_baseline
0,C1,35,1.000000,1.000000,1.000000,1,1
1,C2,23,1.000000,1.000000,1.000000,1,1
2,C3,12,1.000000,0.916667,0.956522,1,1
3,C4,77,0.987179,1.000000,0.993548,1,1
4,C5,61,0.983871,1.000000,0.991870,1,1
5,C6,19,1.000000,1.000000,1.000000,1,1
6,C7,28,1.000000,1.000000,1.000000,1,1
7,C8,10,1.000000,0.900000,0.947368,1,1
8,C9,4,1.000000,1.000000,1.000000,1,1


,manual,klasifikasi,jumlah
0,C3,C5,1
1,C8,C4,1


,model_benar_baseline_salah,model_salah_baseline_benar,keduanya_benar,keduanya_salah,p_mcnemar_eksak,selisih_accuracy
0,0,2,267,0,0.5,-0.007435


,kelompok,n,accuracy_model,accuracy_baseline,model_lebih_benar,baseline_lebih_benar
0,memuat singkatan,75,0.986667,1,0,1
1,tanpa singkatan,194,0.994845,1,0,1


## Q8 sampai Q10: catatan Lainnya, konsistensi, biaya, kalibrasi

In [6]:
display(pd.read_csv(R2/'eval_other_notes.csv')); display(pd.read_csv(R2/'eval_consistency.csv')); display(pd.read_csv(R2/'eval_cost.csv')); display(pd.read_csv(R2/'eval_calibration.csv')); display(pd.read_csv(R2/'eval_threshold.csv'))

,teks,jumlah
0,lihat lot sebelumnya,16
1,"ada masalah, belum tau penyebabnya",14


,sampel,identik_dua_kali_jalan,persen,catatan,informatif_untuk_llm
0,2724,2724,1.0,pengganti lokal deterministik (seed tetap); an...,tidak


,ukuran,nilai,alasan
0,biaya API per 1.000 catatan,tidak diukur,tidak ada API key; Claude API tidak dipanggil
1,waktu per 1.000 catatan (API),tidak diukur,tidak ada API key
2,token per catatan,tidak diukur,tidak ada API key


,rentang,catatan,accuracy,rata_confidence
0,0.00 sampai 0.50,4,0.5,0.367461
1,0.50 sampai 0.70,6,1.0,0.635839
2,0.70 sampai 0.80,14,1.0,0.752846
3,0.80 sampai 0.90,58,1.0,0.863903
4,0.90 sampai 1.00,187,1.0,0.940573


,ambang,diterima_otomatis,coverage,accuracy_otomatis,perlu_cek_manual,salah_lolos,salah_tertangkap,dasar,ambang_alternatif
0,0.3,267,0.992565,1,2,0,2,0,1
1,0.4,267,0.992565,1,2,0,2,0,0
2,0.5,265,0.985130,1,4,0,2,0,0
3,0.6,264,0.981413,1,5,0,2,0,0
4,0.7,259,0.962825,1,10,0,2,1,0
5,0.8,245,0.910781,1,24,0,2,0,0
6,0.9,187,0.695167,1,82,0,2,0,0


## Q11 dan Q12: dampak (estimated)

In [7]:
display(pd.read_csv(R2/'impact_q11.csv')); display(pd.read_csv(R2/'impact_q12.csv')); display(pd.read_csv(R2/'impact_sensitivity.csv'))

,kategori_teratas,nama,catatan_per_bulan,penurunan,menit_per_unit,jam_dihemat_per_bulan,label
0,C4,Perekat kurang / berlebih,124.166667,0.2,12,4.966667,estimated


,catatan_per_bulan,detik_per_catatan,jam_dihemat_per_bulan,label
0,499,20,2.772222,estimated


,analisis,penurunan,parameter,satuan,jam_per_bulan
0,Q11,0.1,6,menit per unit,1.241667
1,Q11,0.1,12,menit per unit,2.483333
2,Q11,0.1,18,menit per unit,3.725000
3,Q11,0.2,6,menit per unit,2.483333
4,Q11,0.2,12,menit per unit,4.966667
5,Q11,0.2,18,menit per unit,7.450000
6,Q11,0.3,6,menit per unit,3.725000
7,Q11,0.3,12,menit per unit,7.450000
8,Q11,0.3,18,menit per unit,11.175000
9,Q11,0.5,6,menit per unit,6.208333


## Q13 dan Q14: kualitas data, Pareto golden, dan cek silang

In [8]:
display(pd.read_csv(R1/'data_quality.csv')); display(pd.read_csv(R2/'eval_pareto_rank.csv')); display(pd.read_csv(R2/'crosscheck_sql_pandas.csv')); display(pd.read_csv(R2/'crosscheck_excel.csv'))

,kategori,jumlah,dasar,proporsi,keputusan
0,catatan (mentah),3060,3060,1.000000,dasar
1,teks kosong,0,3060,0.000000,dibuang
2,catatan ganda,66,3060,0.021569,dibuang (baris pertama dipertahankan)
3,terlalu singkat (< 3 kata),270,3060,0.088235,"dipertahankan, tidak diklasifikasi"
4,catatan setelah dibersihkan,2994,3060,0.978431,dipakai
5,catatan valid untuk klasifikasi,2724,2994,0.909820,dipakai
6,golden set (mentah),300,300,1.000000,dasar
7,golden set valid (dipakai evaluasi),269,300,0.896667,dipakai
8,golden set dibuang (ganda atau terlalu singkat),31,300,0.103333,tidak dipakai evaluasi


,kendall_tau,top3_manual,top3_klasifikasi,urutan_top3_sama,semua_peringkat_sama
0,1,C4 C5 C1,C4 C5 C1,1,1


,pemeriksaan,dibandingkan,cocok,selisih_maks,hasil
0,Pareto kategori (Q1),9,9,4.875167e-11,cocok
1,Pareto per stasiun (Q2),54,54,4.988662e-11,cocok
2,Tren bulanan (Q3),54,54,4.988558e-11,cocok
3,"Golden set: kecocokan label (Q5, Q7)",269,269,0.000000e+00,cocok
4,Confusion matrix (Q6),11,11,0.000000e+00,cocok


,category_code,n_manual_excel,n_manual_sql,n_klasifikasi_excel,n_klasifikasi_sql,rank_manual_excel,rank_manual_sql,rank_klasifikasi_excel,rank_klasifikasi_sql,cocok
0,C1,35,35,35,35,3,3,3,3,1
1,C2,23,23,23,23,5,5,5,5,1
2,C3,12,12,11,11,7,7,7,7,1
3,C4,77,77,78,78,1,1,1,1,1
4,C5,61,61,62,62,2,2,2,2,1
5,C6,19,19,19,19,6,6,6,6,1
6,C7,28,28,28,28,4,4,4,4,1
7,C8,10,10,9,9,8,8,8,8,1
8,C9,4,4,4,4,9,9,9,9,1


## Hipotesis dan cek kebenaran simulasi

In [9]:
display(pd.read_csv(R2/'hypotheses.csv')); display(pd.read_csv(R2/'ground_truth_check.csv'))

,kode,hipotesis,hasil,bukti
0,H1,Dua atau tiga kategori menyumbang sekitar 80% ...,Tidak terbukti,2 kategori 0.4916; 3 kategori (C4 C5 C1) 0.639...
1,H2,"LLM lebih tepat dibanding baseline kata kunci,...",Belum bisa disimpulkan,LLM tidak diuji (tanpa API key); pengganti lok...
2,H3,"Kategori yang mirip, misalnya goresan dan reta...",Belum bisa disimpulkan,total salah 2 pada golden set (minimum 10); te...
3,H4,Satu stasiun menjadi sumber utama untuk satu k...,Terbukti,4 dari 54 sel memenuhi rasio >= 1.5 dan Holm p...
4,H5,Sekitar 5 sampai 10% catatan terlalu singkat a...,Terbukti,270 dari 2994 catatan (0.0902; CI Wilson 0.080...


,fakta,ditanam,terdeteksi,hasil
0,ST-C menyumbang lebih banyak C4,pengganda 2 sampai 4 kali,rasio 1.87; Holm p 3.52e-23,Sesuai
1,ST-E menyumbang lebih banyak C5,pengganda 2 sampai 4 kali,rasio 2.06; Holm p 5.38e-23,Sesuai
2,ST-D menyumbang lebih banyak C7,pengganda 2 sampai 4 kali,rasio 2.41; Holm p 1.98e-16,Sesuai
3,ST-A menyumbang lebih banyak C2,pengganda 2 sampai 4 kali,rasio 1.92; Holm p 1.11e-08,Sesuai
4,Kategori teratas pada Pareto adalah C4,"bobot dasar C4 0,23 tertinggi",teratas C4; kebenaran C4,Sesuai
5,Urutan 5 kategori teratas sama dengan kebenaran,bobot dasar dan pengganda stasiun,hasil C4 C5 C1 C7 C2; kebenaran C4 C5 C1 C7 C2,Sesuai
6,Tidak ada efek shift dan model,tidak ada,p Holm shift 1.000; model 1.000,Sesuai
7,Tidak ada lonjakan bulanan yang ditanam,tidak ada,0 lonjakan dan 0 penurunan melewati 3 sigma,Sesuai
8,Sekitar 9% catatan terlalu singkat,"75% dari C9 (bobot 0,12) dibuat sangat singkat",0.0902 dari catatan setelah dibersihkan,Sesuai
9,Ketepatan klasifikasi terhadap kebenaran gener...,teks dari templat sederhana,pengganti lokal 0.9993; baseline 1.0000,Sesuai
